# Imports

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager 
from tqdm import tqdm
from bs4 import BeautifulSoup
import time 
import os
from pathlib import Path 

In [ ]:
bible_languages_url = {
    "english": "https://www.bible.com/bible/37/GEN.1.NIV",              #1
    "tagalog": "https://www.bible.com/bible/2195/GEN.1.ABTAG01",        #11
    "cebuano": "https://www.bible.com/bible/2187/GEN.1.ABCEB",       #14
    "waray": "https://www.bible.com/bible/2198/GEN.1.MBBSAM",           #15
}

bible_books = ["GEN", "EXO", "LEV", "NUM", "DEU", 
               "MAT", "MRK", "LUK", "JHN", "ACT", 
               "ROM", "1C0", "2CO", "GAL", "EPH", 
               "PHP", "COL", "1TH", "2TH", "1TI", 
               "2TI", "TIT", "PHM", "HEB", "JAS", 
               "1PE", "2PE", "1JN", "2JN", "3JN", 
               "JUD", "REV"]

In [3]:
print("Setting up WebDriver...")
options = Options()
options.add_argument("--headless")  # run chrome without opening a visual window
options.add_argument("--log-level=3")  # suppress unnecessary logs
options.add_experimental_option('excludeSwitches', ['enable-logging'])

# use WebDriver Manager to handle driver installation/updates automatically
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
print("WebDriver ready.")

# data structures for storing scraped data and statistics
bible_data_new = {}  # {lang: {book: {chapter: [verses]}}}
word_counts_new = {}  # {lang: count}
total_words_new = 0
total_verses_new = 0
total_chapters_new = 0

# progress bar setup
total_iterations = len(bible_languages_url) * len(bible_books)
pbar = tqdm(total=total_iterations, desc="Overall Progress", unit="book")

try:
    for lang, root_url in bible_languages_url.items():
        bible_data_new[lang] = {}
        word_counts_new[lang] = 0

        # extract the parts of the URL
        try:
            parts = root_url.split("/")
            base_bible_url = f"https://www.bible.com/bible/{parts[4]}"
            version = root_url.split(".")[-1]
        except IndexError:
            print(f"Skipping invalid URL format for {lang}: {root_url}")
            pbar.update(len(bible_books))  
            continue  # skip to next language if error occurs

        for book in bible_books:
            pbar.set_description(f"Scraping {lang} - {book}")
            bible_data_new[lang][book] = {}

            ch = 1
            while True:
                url = f"{base_bible_url}/{book}.{ch}.{version}"
                driver.get(url)
                time.sleep(1)  # wait for page to load

                soup = BeautifulSoup(driver.page_source, "html.parser")
                
                # checks if end of chapters reached using "not available" marker
                not_available = soup.find("span", class_="ChapterContent_not-avaliable-span__WrOM_")
                if not_available:
                    break

                # if not yet end of chapters, extract
                chapter_content = soup.find_all("span", {"data-usfm": True})

                # mark as empty if no content found
                if not chapter_content:
                    bible_data_new[lang][book][ch] = ["MISSING"]
                else:
                    chapter_verses_new = []
                    chapter_word_count_new = 0

                    # Group elements by data-usfm to handle duplicate verse codes
                    verse_groups = {}
                    for verse in chapter_content:
                        usfm = verse.get("data-usfm")
                        if usfm not in verse_groups:
                            verse_groups[usfm] = []
                        verse_groups[usfm].append(verse)

                    # Process each verse group
                    for usfm, verse_elements in verse_groups.items():
                        combined_text_parts = []
                        
                        for verse in verse_elements:
                            # remove footnotes within the verse
                            for note in verse.find_all("span", class_=lambda x: x and x.startswith("ChapterContent_note")):
                                note.decompose()

                            # extract clean verse text
                            verse_text = verse.get_text(" ", strip=True)
                            if verse_text:
                                combined_text_parts.append(verse_text)
                        
                        # Combine all parts of the same verse
                        if combined_text_parts:
                            combined_verse_text = " ".join(combined_text_parts)
                            chapter_verses_new.append(combined_verse_text)
                            verse_words_new = len(combined_verse_text.split())
                            chapter_word_count_new += verse_words_new
                            total_verses_new += 1

                    # add verses
                    bible_data_new[lang][book][ch] = (
                        chapter_verses_new if chapter_verses_new else ["MISSING"]
                    )

                    # update word counts
                    word_counts_new[lang] += chapter_word_count_new
                    total_words_new += chapter_word_count_new
                    total_chapters_new += 1

                    pbar.set_postfix({
                        'Words': f"{total_words_new:,}",
                        'Verses': f"{total_verses_new:,}",
                        'Chapters': total_chapters_new
                    })

                ch += 1       # next chapter
                if ch > 100:  # safety cap
                    break

            pbar.update(1)

finally:
    driver.quit()
    pbar.close()
    print("Scraping complete.")

# summary statistics
print("\n" + "="*60)
if total_verses_new > 0 and total_chapters_new > 0:
    print(f"Total Words: {total_words_new:,}")
    print(f"Total Verses: {total_verses_new:,}")
    print(f"Total Chapters: {total_chapters_new}")
    print(f"Average Words per Verse: {total_words_new/total_verses_new:.1f}")
    print(f"Average Words per Chapter: {total_words_new/total_chapters_new:.1f}")
else:
    print("No data scraped or processed.")

print("\nWord Count by Language:")
print("-" * 30)
if total_words_new > 0:
    for lang_key, count in word_counts_new.items():
        if count > 0:
            percentage = (count / total_words_new) * 100
            print(f"{lang_key:12}: {count:8,} words ({percentage:.1f}%)")
        else:
            print(f"{lang_key:12}: {count:8,} words (0.0%) - Check availability/URL")
else:
    print("No words counted.")

Setting up WebDriver...
WebDriver ready.
WebDriver ready.


Scraping english - GEN:   0%|          | 0/128 [00:29<?, ?book/s, Words=12,909, Verses=592, Chapters=23]



Scraping complete.


KeyboardInterrupt: 

In [ ]:
bible_data_new

{'english': {'MAT': {1: ['1 A record of the ancestors of Jesus Christ, son of David, son of Abraham:',
    '2 Abraham was the father of Isaac.',
    'Isaac was the father of Jacob.',
    'Jacob was the father of Judah and his brothers.',
    '3 Judah was the father of Perez and Zerah,',
    'whose mother was Tamar.',
    'Perez was the father of Hezron.',
    'Hezron was the father of Aram.',
    '4 Aram was the father of Amminadab.',
    'Amminadab was the father of Nahshon.',
    'Nahshon was the father of Salmon.',
    '5 Salmon was the father of Boaz, whose mother was Rahab.',
    'Boaz was the father of Obed, whose mother was Ruth.',
    'Obed was the father of Jesse.',
    '6 Jesse was the father of David the king.',
    'David was the father of Solomon,',
    'whose mother had been the wife of Uriah.',
    '7 Solomon was the father of Rehoboam.',
    'Rehoboam was the father of Abijah.',
    'Abijah was the father of Asaph.',
    '8 Asaph was the father of Jehoshaphat.',
    'Je

In [ ]:
# Save bible data to text files
output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

for lang, books in bible_data_new.items():
    output_file = output_dir / f"{lang}_raw.txt"
    
    with open(output_file, "w", encoding="utf-8") as f:
        for book, chapters in books.items():
            for chapter_num, verses in chapters.items():
                for verse_num, verse_text in enumerate(verses, start=1):
                    # Format: Book Chapter:Verse Text
                    f.write(f"{book} {chapter_num}:{verse_num} {verse_text}\n")
    
    print(f"Saved {lang} data to {output_file}")

print("\nAll files saved successfully!")

Saved english data to ..\data\raw\english_raw.txt
Saved tagalog data to ..\data\raw\tagalog_raw.txt
Saved cebuano data to ..\data\raw\cebuano_raw.txt
Saved waray data to ..\data\raw\waray_raw.txt

All files saved successfully!
